# 02 — Iterative BOM Decomposition (Bottom-Up Arm)\n\nStarting from an R&S spectrum monitoring product, decompose level by level:\n\n**Product → Subsystem → Component → Sub-component → Fundamental Technology**\n\nEach level = one focused LLM call with only relevant RAG chunks. Leaf nodes get classified as Tech Drivers or not.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.config import CHROMA_PERSIST_DIR, MAX_RAG_CHUNKS, BOM_MAX_DEPTH
from src.llm import embed, safe_chat_json
from src.traceability import BOMNode, TechDriver, DriverOrigin, DriverConfidence, KBPool, stable_id
from src import prompts

import chromadb

client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
collection = client.get_collection("knowledge_base")

# load KB state from notebook 01
with open("../data/outputs/kb_state.json") as f:
    kb_state = json.load(f)

print(f"KB: {len(kb_state['sources'])} sources, {len(kb_state['chunks'])} chunks")
print(f"ChromaDB: {collection.count()} documents")

KB: 45 sources, 2850 chunks
ChromaDB: 2850 documents


In [2]:
def retrieve_product(query: str, n: int = MAX_RAG_CHUNKS) -> list[dict]:
    """Retrieve from product pool only."""
    query_emb = embed([query])[0]
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n,
        where={"pool": "product"},
        include=["documents", "metadatas"],
    )
    out = []
    for i in range(len(results["ids"][0])):
        out.append({
            "chunk_id": results["ids"][0][i],
            "content": results["documents"][0][i],
            "source_title": results["metadatas"][0][i]["source_title"],
        })
    return out


def format_rag_chunks(chunks: list[dict]) -> str:
    """Format chunks for prompt injection with chunk IDs for traceability."""
    parts = []
    for c in chunks:
        parts.append(f"[Chunk ID: {c['chunk_id']}] (Source: {c['source_title']})\n{c['content']}")
    return "\n\n---\n\n".join(parts)


# store all BOM nodes
bom_nodes: dict[str, BOMNode] = {}


def decompose(node: BOMNode, max_depth: int = BOM_MAX_DEPTH) -> BOMNode:
    """Recursively decompose a BOM node into sub-components."""
    if node.level >= max_depth:
        return node

    # RAG: retrieve chunks relevant to this specific component
    query = f"{node.name} {node.description} components sub-components technology"
    rag_chunks = retrieve_product(query, n=3)
    rag_text = format_rag_chunks(rag_chunks)

    # build parent context
    parent_context = "Top-level product" if node.parent_id is None else f"Part of: {bom_nodes[node.parent_id].name}"

    prompt = prompts.BOM_DECOMPOSE.format(
        parent_context=parent_context,
        component_name=node.name,
        component_description=node.description,
        rag_chunks=rag_text,
    )

    response = safe_chat_json(prompt, system="You are a technical analyst decomposing spectrum monitoring equipment into sub-components.")

    for comp in response.get("components", []):
        child = BOMNode(
            id=stable_id(node.id, comp["name"]),
            name=comp["name"],
            description=comp.get("description", ""),
            level=node.level + 1,
            parent_id=node.id,
            source_chunk_ids=response.get("source_chunk_ids_used", []),
        )
        bom_nodes[child.id] = child
        node.children_ids.append(child.id)

        indent = "  " * child.level
        leaf_marker = " 🔎" if comp.get("is_leaf") else ""
        print(f"{indent}├── {child.name}{leaf_marker}")

        if not comp.get("is_leaf", False):
            decompose(child, max_depth)

    return node

## Run Decomposition: R&S ESMW Ultra Wideband Monitoring Receiver

In [3]:
# start from the ESMW as our target product
root = BOMNode(
    id=stable_id("root", "R&S ESMW Ultra Wideband Monitoring Receiver"),
    name="R&S ESMW Ultra Wideband Monitoring Receiver",
    description="Next generation ultra wideband monitoring receiver for spectrum monitoring and direction finding. Frequency range 8 kHz to 40 GHz, up to 2 GHz real-time bandwidth, panorama scan at 2.6 THz/s, 75ns minimum signal duration for 100% POI, I/Q streaming via 100GE, AoA direction finding, TDOA localization, ITU-compliant.",
    level=0,
)
bom_nodes[root.id] = root

print(f"ROOT: {root.name} (id={root.id})\n")
decompose(root, max_depth=3)
print(f"\nTotal BOM nodes: {len(bom_nodes)}")

ROOT: R&S ESMW Ultra Wideband Monitoring Receiver (id=b03b55dda54d)



  ├── Base Unit Receiver


    ├── RF Frontend


      ├── Preselection Stage
      ├── High-Gain Preamplifier 🔎
      ├── Frequency Conversion
    ├── Real-Time Spectrum Analyzer


      ├── Analog-to-Digital Converter (ADC) 🔎
      ├── Resampler / Digital Down Converter (DDC)
      ├── FFT Processing Unit (N bin FFT)
      ├── Point Reduction Unit (N → M)
      ├── IQ Memory
      ├── Detector
      ├── Persistence Matrix and Persistence Spectrum
      ├── Frequency Mask Trigger (FMT) Evaluation
      ├── Trigger Control
      ├── Real-Time FPGA and ASIC Pipeline
      ├── CPU and Display Interface
    ├── Digital Downconverters


...

    ├── Real-Time Spectrum Path


      ├── Wideband Analog-to-Digital Converter (ADC) 🔎
      ├── Fast Fourier Transform (FFT) Processing Engine 🔎
      ├── Parallel Digital Receive Path Architecture
      ├── Selectable Resolution Bandwidth Filters 🔎
      ├── Preselection Filtering and Switched Filter Banks 🔎
      ├── Spectrum Trace and Detection Modes 🔎
    ├── Bandwidth Extension Hardware Modules


      ├── Real-time spectrum path
      ├── Bandwidth Extension Hardware Module R&S®HS1-BW500 🔎
      ├── Bandwidth Extension Firmware Option R&S®HS1-BW2000 🔎
      ├── High-quality active components and preselection filtering 🔎
    ├── Selectable Resolution Bandwidth Filters 🔎
  ├── R&S CS-MC53 Microwave Converter 🔎
  ├── Digital Downconverters (DDC) 🔎
  ├── I/Q Data Streaming Interfaces 🔎
  ├── Angle of Arrival (AoA) Direction Finding Module


    ├── Correlative Interferometer Direction Finding Method 🔎
    ├── Single-Channel DF Antennas (e.g., R&S®ADD597, R&S®ADD119) 🔎
    ├── Wideband Direction Finding Capability 🔎
    ├── DF Signal Processing and Display Module 🔎
    ├── Multi-Receiver Network and Triangulation


      ├── Multiple DF Stations 🔎
      ├── Cross Bearing Calculation 🔎
      ├── Data Link Infrastructure 🔎
      ├── Measurement Data Processing and Triangulation Engine
    ├── Preselection and Phase Noise Minimization Subsystem 🔎
  ├── TDOA Radiolocation Module 🔎
  ├── Rohde & Schwarz Single-Channel Direction Finding Antennas 🔎
  ├── Panorama Scan Functionality 🔎

Total BOM nodes: 74


## Classify Leaf Nodes as Tech Drivers

In [4]:
def get_bom_path(node_id: str) -> str:
    """Trace path from root to this node."""
    path = []
    current = bom_nodes[node_id]
    while current:
        path.append(current.name)
        current = bom_nodes.get(current.parent_id) if current.parent_id else None
    return " → ".join(reversed(path))


# find leaf nodes (no children)
leaves = [n for n in bom_nodes.values() if not n.children_ids]
print(f"Found {len(leaves)} leaf nodes to classify\n")

bom_drivers: list[TechDriver] = []

for leaf in leaves:
    bom_path = get_bom_path(leaf.id)
    prompt = prompts.BOM_CLASSIFY_DRIVER.format(
        name=leaf.name,
        description=leaf.description,
        bom_path=bom_path,
    )
    result = safe_chat_json(prompt, system="You are a technology analyst evaluating whether components represent active technology drivers.")

    leaf.is_tech_driver = result.get("is_tech_driver", False)
    marker = "✓ DRIVER" if leaf.is_tech_driver else "✗ passive"
    print(f"  [{marker}] {leaf.name} — {result.get('reasoning', '')[:80]}")

    if leaf.is_tech_driver:
        driver = TechDriver(
            id=stable_id("bom", leaf.name),
            name=leaf.name,
            description=f"{leaf.description}. {result.get('reasoning', '')}",
            origin=DriverOrigin.BOM,
            confidence=DriverConfidence.MEDIUM,
            bom_node_id=leaf.id,
            source_chunk_ids=leaf.source_chunk_ids,
        )
        bom_drivers.append(driver)

print(f"\n=== {len(bom_drivers)} Tech Drivers identified from BOM ===")

Found 60 leaf nodes to classify



  [✗ passive] Preselection Stage — The preselection stage, involving bandpass and tracking filters to suppress out-


  [✗ passive] High-Gain Preamplifier — High-gain preamplifiers are a well-established technology component in RF system


  [✗ passive] Frequency Conversion — Frequency conversion is a well-established and mature technology fundamental to 


  [✓ DRIVER] Analog-to-Digital Converter (ADC) — Analog-to-Digital Converters (ADCs) remain a critical technology driver in high-


...

  [✓ DRIVER] Measurement Data Processing and Triangulation Engine — The Measurement Data Processing and Triangulation Engine is a critical component


  [✓ DRIVER] Preselection and Phase Noise Minimization Subsystem — The Preselection and Phase Noise Minimization Subsystem addresses critical chall


  [✓ DRIVER] TDOA Radiolocation Module — TDOA (Time Difference of Arrival) radiolocation combined with GNSS modules is an


  [✗ passive] Rohde & Schwarz Single-Channel Direction Finding Antennas — The Rohde & Schwarz Single-Channel Direction Finding Antennas are mature, well-e


  [✓ DRIVER] Panorama Scan Functionality — Panorama Scan Functionality enables extremely fast spectrum scanning speeds (up 

=== 28 Tech Drivers identified from BOM ===


In [5]:
# deduplicate drivers by name (merge source_chunk_ids)
seen: dict[str, TechDriver] = {}
for d in bom_drivers:
    key = d.name.lower().strip()
    if key in seen:
        for sid in d.source_chunk_ids:
            if sid not in seen[key].source_chunk_ids:
                seen[key].source_chunk_ids.append(sid)
    else:
        seen[key] = d

bom_drivers = list(seen.values())
print(f"After dedup: {len(bom_drivers)} unique BOM drivers\n")
for d in bom_drivers:
    print(f"  • {d.name} ({len(d.source_chunk_ids)} sources)")

After dedup: 27 unique BOM drivers

  • Analog-to-Digital Converter (ADC) (3 sources)
  • Resampler / Digital Down Converter (DDC) (3 sources)
  • Detector (3 sources)
  • Real-Time FPGA and ASIC Pipeline (3 sources)
  • Parallel Digital Downconverters (1 sources)
  • Digital Downconverter (DDC) (2 sources)
  • R&S®HS1-MIQ multi I/Q interface option (1 sources)
  • Baseband I/Q streaming paths (1 sources)
  • R&S CS-MC53 microwave converter (4 sources)
  • Wideband Analog-to-Digital Converter (ADC) (3 sources)
  • Fast Fourier Transform (FFT) Processing Engine (3 sources)
  • Parallel Digital Receive Path Architecture (3 sources)
  • Spectrum Trace and Detection Modes (3 sources)
  • Real-time spectrum path (3 sources)
  • Bandwidth Extension Hardware Module R&S®HS1-BW500 (3 sources)
  • Bandwidth Extension Firmware Option R&S®HS1-BW2000 (3 sources)
  • High-quality active components and preselection filtering (3 sources)
  • Digital Downconverters (DDC) (3 sources)
  • I/Q Data Stream

## Visualize BOM Tree & Save State

In [6]:
def print_tree(node_id: str, prefix: str = ""):
    node = bom_nodes[node_id]
    tag = ""
    if node.is_tech_driver:
        tag = " ⚡ DRIVER"
    elif not node.children_ids:
        tag = " (leaf)"
    print(f"{prefix}{node.name}{tag}")
    for i, child_id in enumerate(node.children_ids):
        is_last = i == len(node.children_ids) - 1
        child_prefix = prefix + ("└── " if is_last else "├── ")
        next_prefix = prefix + ("    " if is_last else "│   ")
        child = bom_nodes[child_id]
        child_tag = ""
        if child.is_tech_driver:
            child_tag = " ⚡ DRIVER"
        elif not child.children_ids:
            child_tag = " (leaf)"
        print(f"{child_prefix}{child.name}{child_tag}")
        if child.children_ids:
            for j, grandchild_id in enumerate(child.children_ids):
                gc_is_last = j == len(child.children_ids) - 1
                gc_prefix = next_prefix + ("└── " if gc_is_last else "├── ")
                gc_next = next_prefix + ("    " if gc_is_last else "│   ")
                gc = bom_nodes[grandchild_id]
                gc_tag = " ⚡ DRIVER" if gc.is_tech_driver else (" (leaf)" if not gc.children_ids else "")
                print(f"{gc_prefix}{gc.name}{gc_tag}")
                if gc.children_ids:
                    for k, ggc_id in enumerate(gc.children_ids):
                        ggc_last = k == len(gc.children_ids) - 1
                        ggc_prefix = gc_next + ("└── " if ggc_last else "├── ")
                        ggc = bom_nodes[ggc_id]
                        ggc_tag = " ⚡ DRIVER" if ggc.is_tech_driver else " (leaf)"
                        print(f"{ggc_prefix}{ggc.name}{ggc_tag}")

print("=== BOM TREE ===\n")
print_tree(root.id)

=== BOM TREE ===

R&S ESMW Ultra Wideband Monitoring Receiver
├── Base Unit Receiver
│   ├── RF Frontend
│   │   ├── Preselection Stage (leaf)
│   │   ├── High-Gain Preamplifier (leaf)
│   │   └── Frequency Conversion (leaf)
│   ├── Real-Time Spectrum Analyzer
│   │   ├── Analog-to-Digital Converter (ADC) ⚡ DRIVER
│   │   ├── Resampler / Digital Down Converter (DDC) ⚡ DRIVER
│   │   ├── FFT Processing Unit (N bin FFT) (leaf)
│   │   ├── Point Reduction Unit (N → M) (leaf)
│   │   ├── IQ Memory (leaf)
│   │   ├── Detector ⚡ DRIVER
│   │   ├── Persistence Matrix and Persistence Spectrum (leaf)
│   │   ├── Frequency Mask Trigger (FMT) Evaluation (leaf)
│   │   ├── Trigger Control (leaf)
│   │   ├── Real-Time FPGA and ASIC Pipeline ⚡ DRIVER
│   │   └── CPU and Display Interface (leaf)
│   ├── Digital Downconverters
│   │   ├── Independent Parameter Settings (leaf)
│   │   └── Parallel Digital Downconverters ⚡ DRIVER
│   ├── Demodulation Module
│   │   ├── Digital Downconverter (DDC) ⚡ DRIV

In [7]:
# save state for downstream notebooks
bom_state = {
    "bom_nodes": {k: v.model_dump(mode="json") for k, v in bom_nodes.items()},
    "bom_drivers": [d.model_dump(mode="json") for d in bom_drivers],
    "root_id": root.id,
}
with open("../data/outputs/bom_state.json", "w") as f:
    json.dump(bom_state, f, indent=2)

print(f"Saved {len(bom_nodes)} BOM nodes, {len(bom_drivers)} tech drivers")
print(f"\nBOM Drivers:")
for d in bom_drivers:
    print(f"  [{d.confidence.value}] {d.name}")
    print(f"    Sources: {d.source_chunk_ids[:3]}")
    print()

Saved 74 BOM nodes, 27 tech drivers

BOM Drivers:
  [medium] Analog-to-Digital Converter (ADC)
    Sources: ['9f2adcd1a377', 'dbde34a1f6d8', '213b35d1da5e']

  [medium] Resampler / Digital Down Converter (DDC)
    Sources: ['9f2adcd1a377', 'dbde34a1f6d8', '213b35d1da5e']

  [medium] Detector
    Sources: ['9f2adcd1a377', 'dbde34a1f6d8', '213b35d1da5e']

  [medium] Real-Time FPGA and ASIC Pipeline
    Sources: ['9f2adcd1a377', 'dbde34a1f6d8', '213b35d1da5e']

  [medium] Parallel Digital Downconverters
    Sources: ['059ffbeeb633']

  [medium] Digital Downconverter (DDC)
    Sources: ['d932ea82c126', '059ffbeeb633']

  [medium] R&S®HS1-MIQ multi I/Q interface option
    Sources: ['013522f14ab8']

  [medium] Baseband I/Q streaming paths
    Sources: ['013522f14ab8']

  [medium] R&S CS-MC53 microwave converter
    Sources: ['34ed6ec97fdf', 'edd727651d08', 'bb9c528e7216']

  [medium] Wideband Analog-to-Digital Converter (ADC)
    Sources: ['56102a63603c', 'd932ea82c126', '9f2adcd1a377']

  